## 0. Setup and Configuration

In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
# Journal reproducibility environment pins
# This cell writes a pinned requirements file and reports installed versions.
# The latest XGBoost and SHAP releases require Python 3.12 or newer.
import sys
from pathlib import Path
import importlib.metadata as importlib_metadata

LATEST_PACKAGE_PINS = {
    "numpy": "2.5.1",
    "pandas": "3.0.3",
    "scipy": "1.18.0",
    "scikit-learn": "1.9.0",
    "imbalanced-learn": "0.14.2",
    "xgboost": "3.3.0",
    "lightgbm": "4.6.0",
    "catboost": "1.2.10",
    "shap": "0.52.0",
    "lime": "0.2.0.1",
    "matplotlib": "3.11.0",
    "seaborn": "0.13.2",
    "openpyxl": "3.1.5",
    "joblib": "1.5.3",
    "threadpoolctl": "3.6.0",
}
req_text = "# Recommended Python runtime: 3.12\n" + "\n".join(
    f"{pkg}=={ver}" for pkg, ver in LATEST_PACKAGE_PINS.items()
) + "\n"
Path("requirements_latest_python312.txt").write_text(req_text, encoding="utf-8")
print("Wrote requirements_latest_python312.txt")
print(f"Current Python: {sys.version.split()[0]}")
print("Pinned package versions for journal reproduction:")
for pkg, target in LATEST_PACKAGE_PINS.items():
    try:
        installed = importlib_metadata.version(pkg)
    except importlib_metadata.PackageNotFoundError:
        installed = "not installed"
    status = "OK" if installed == target else "check"
    print(f"  {pkg:18s} target={target:10s} installed={installed:12s} {status}")


Wrote requirements_latest_python312.txt
Current Python: 3.12.13
Pinned package versions for journal reproduction:
  numpy              target=2.5.1      installed=2.0.2        check
  pandas             target=3.0.3      installed=2.3.3        check
  scipy              target=1.18.0     installed=1.16.3       check
  scikit-learn       target=1.9.0      installed=1.6.1        check
  imbalanced-learn   target=0.14.2     installed=0.14.1       check
  xgboost            target=3.3.0      installed=3.2.0        check
  lightgbm           target=4.6.0      installed=4.6.0        OK
  catboost           target=1.2.10     installed=1.2.10       OK
  shap               target=0.52.0     installed=0.51.0       check
  lime               target=0.2.0.1    installed=0.2.0.1      OK
  matplotlib         target=3.11.0     installed=3.10.0       check
  seaborn            target=0.13.2     installed=0.13.2       OK
  openpyxl           target=3.1.5      installed=3.1.5        OK
  joblib         

In [3]:
%%writefile config.py
"""Central configuration for the single dataset CKD explainable ML notebook."""

import os
from pathlib import Path

ROOT = Path(os.environ.get(
    "CKD_ROOT",
    "/kaggle/working" if Path("/kaggle/working").exists() else Path.cwd()
)).resolve()


def _resolve_data_file(env_var, preferred, names):
    env_value = os.environ.get(env_var)
    if env_value:
        env_path = Path(env_value).expanduser()
        if env_path.exists():
            return env_path.resolve()
    preferred = Path(preferred)
    candidates = [preferred]
    for base in [ROOT, Path.cwd(), Path("/mnt/data")]:
        for name in names:
            candidates.append(base / name)
    for path in candidates:
        if path.exists():
            return path.resolve()
    for base in [Path("/kaggle/input"), ROOT, Path.cwd(), Path("/mnt/data")]:
        if not base.exists():
            continue
        for name in names:
            matches = sorted(base.rglob(name))
            if matches:
                return matches[0].resolve()
    return preferred


DATA = _resolve_data_file("CKD_DATA", "/kaggle/input/datasets/miftahuladib/datasets-ckd/CKD_dataset.xlsx", ["CKD_dataset.xlsx", "ckd_dataset.xlsx"])
OUT = ROOT / "outputs_urinalysis"
FIGDIR = OUT / "figures"
TABDIR = OUT / "tables"
ARTDIR = OUT / "artifacts"
for _d in (OUT, FIGDIR, TABDIR, ARTDIR):
    _d.mkdir(parents=True, exist_ok=True)

DATASET_TITLE = "Dataset 1, Urinalysis CKD"
DATASET_LABEL = "Dataset 1 (Urinalysis)"
REMOVED_PREDICTORS = ["p_id"]
DERIVED_PREDICTORS = []
TARGET_DESCRIPTION = "d_status converted to CKD equals 1 and not CKD equals 0"
GROUP_COLUMN_DESCRIPTION = None
VALIDATION_SUMMARY = "Stratified 5 by 3 nested cross validation"
PREPROCESSING_SUMMARY = "Preprocessing inside each fold: categorical encoding, median and mode imputation, scaling for logistic regression, SVM, and MLP, class weighting"
MODEL_SUMMARY = "8 models: logistic regression, decision tree, random forest, XGBoost, LightGBM, CatBoost, SVM, and MLP"
XAI_SUMMARY = "Model selection, SHAP stability, permutation importance, PDP and ICE, local SHAP and LIME, learning curve"
PERFORMANCE_TABLE_NAME = "table2_dataset1_performance"
DESCRIPTIVE_TABLE_NAME = "supp_descriptive_dataset1"

PROFILE = os.environ.get("CKD_PROFILE", "paper").lower()
RNG = 42
DPI = 300

if PROFILE == "smoke":
    SEEDS = [0]
    N_ITER_SIMPLE = 1
    N_ITER_BOOST = 1
    OUTER_SPLITS = 2
    INNER_SPLITS = 2
    N_BOOTSTRAP = 10
    SUBSAMPLE_PATIENTS = None
    MODELS = ["logreg", "decision_tree"]
    PERM_REPEATS = 1
    N_JOBS = 1
else:
    SEEDS = [0, 1, 2]
    N_ITER_SIMPLE = 15
    N_ITER_BOOST = 40
    OUTER_SPLITS = 5
    INNER_SPLITS = 3
    N_BOOTSTRAP = 1000
    SUBSAMPLE_PATIENTS = None
    MODELS = [
        "logreg", "decision_tree", "random_forest", "xgboost",
        "lightgbm", "catboost", "svm", "mlp",
    ]
    PERM_REPEATS = 10
    N_JOBS = 1

MODEL_LABELS = {
    "logreg": "Logistic Regression",
    "decision_tree": "Decision Tree",
    "random_forest": "Random Forest",
    "xgboost": "XGBoost",
    "lightgbm": "LightGBM",
    "catboost": "CatBoost",
    "svm": "SVM (RBF)",
    "mlp": "MLP",
}
SCALED_MODELS = {"logreg", "svm", "mlp"}
TREE_MODELS = {"decision_tree", "random_forest", "xgboost", "lightgbm", "catboost"}
REPORT_SEED = 0

PYTHON_RECOMMENDED = "3.12"
PACKAGE_PIN_DATE = "2026-07-06"
LATEST_PACKAGE_PINS = {
    "numpy": "2.5.1", "pandas": "3.0.3", "scipy": "1.18.0",
    "scikit-learn": "1.9.0", "imbalanced-learn": "0.14.2",
    "xgboost": "3.3.0", "lightgbm": "4.6.0", "catboost": "1.2.10",
    "shap": "0.52.0", "lime": "0.2.0.1", "matplotlib": "3.11.0",
    "seaborn": "0.13.2", "openpyxl": "3.1.5", "joblib": "1.5.3",
    "threadpoolctl": "3.6.0",
}


def latest_requirements_text():
    lines = [
        f"# Generated for this CKD journal notebook on {PACKAGE_PIN_DATE}",
        f"# Recommended Python runtime: {PYTHON_RECOMMENDED}",
    ]
    lines.extend(f"{name}=={version}" for name, version in LATEST_PACKAGE_PINS.items())
    return "\n".join(lines) + "\n"


print(f"CKD profile: {PROFILE}")
print(f"Dataset path: {DATA}")
print(f"Output path: {OUT}")
print(f"N_JOBS: {N_JOBS}")

Writing config.py


## 1. Data Loading and Preprocessing

In [4]:
%%writefile data.py
"""Loading and preparation for the urinalysis CKD dataset only."""

from dataclasses import dataclass
import numpy as np
import pandas as pd
import config


def _require_columns(df, columns, dataset_name):
    missing = [c for c in columns if c not in df.columns]
    if missing:
        raise KeyError(f"{dataset_name} is missing required columns: {missing}")


def _to_numeric(series):
    return pd.to_numeric(series, errors="coerce")


@dataclass
class Dataset:
    name: str
    X: pd.DataFrame
    y: np.ndarray
    groups: np.ndarray | None
    continuous: list
    categorical: list
    cont_idx: list
    cat_idx: list
    primary: str
    grouped: bool
    raw: pd.DataFrame
    feature_categories: dict


FEATURE_CATEGORY = {
    "age": "Demographic", "gender": "Demographic",
    "ph": "Urinalysis", "sp_g": "Urinalysis", "u_clarity": "Urinalysis",
    "albumin": "Urinalysis", "glucose": "Urinalysis", "sugar": "Urinalysis",
    "kb": "Urinalysis", "bpigment": "Urinalysis", "ur_bi": "Urinalysis",
    "blood": "Urinalysis", "pus_cells": "Urinalysis", "red_cells": "Urinalysis",
    "epi_cells": "Urinalysis", "mt": "Urinalysis", "co": "Urinalysis",
    "gc": "Urinalysis", "bacteria": "Urinalysis", "cc": "Urinalysis",
}


def load_dataset():
    raw = pd.read_excel(config.DATA)
    df = raw.copy()
    continuous = ["age", "ph", "sp_g"]
    _require_columns(df, ["p_id", "d_status"] + continuous, "Urinalysis dataset")
    df = df.drop(columns=["p_id"])
    y = (df["d_status"].astype(str).str.strip().str.lower() == "ckd").astype(int).values
    df = df.drop(columns=["d_status"])
    categorical = [c for c in df.columns if c not in continuous]
    for c in categorical:
        df[c] = df[c].astype(str).str.strip().str.lower().astype("category").cat.codes
    for c in continuous:
        df[c] = _to_numeric(df[c])
    X = df[continuous + categorical].reset_index(drop=True)
    return Dataset(
        name="Dataset1_Urinalysis", X=X, y=y, groups=None,
        continuous=continuous, categorical=categorical,
        cont_idx=[X.columns.get_loc(c) for c in continuous],
        cat_idx=[X.columns.get_loc(c) for c in categorical],
        primary="roc_auc", grouped=False, raw=raw,
        feature_categories=FEATURE_CATEGORY,
    )


Writing data.py


## 2. Modeling

In [5]:
%%writefile modeling.py
"""
Modeling core: estimators, per-family preprocessing pipelines, hyperparameter
search spaces, the nested cross-validation engine, metrics, patient-level
bootstrap confidence intervals, calibration metrics, and the model-selection rule.

Key design points (matching the locked plan):
  * All preprocessing (scaling, resampling) is fit inside training folds only.
  * Nested CV: outer folds give the reported estimate, inner folds tune
    hyperparameters and the decision threshold (by MCC).
  * Grouped datasets use patient-grouped folds (StratifiedGroupKFold) so no patient
    appears in both train and test.
  * Imbalance handling: class weighting is the primary strategy; SMOTENC is a
    separate comparison. MLP cannot take class weights in scikit-learn, so it
    uses random oversampling of the training fold as its weighting mechanism.

Fixes applied (v2):
  FIX 1 (critical)      – Threshold computation moved OUTSIDE the inner fold
                          loop so best_mcc_threshold sees fully-populated
                          inner_proba, not a partially-NaN array.
  FIX 2 (significant)   – LightGBM now receives is_unbalance=True when
                          imbalance="balanced", matching every other model.
  FIX 3 (reproducibility) – CatBoost thread_count=1 for cross-machine
                          bit-exact results (was using all available cores).
  FIX 4 (reproducibility) – bootstrap_ci accepts a seed parameter so each
                          call uses independent random draws.
  FIX 5 (minor)         – select_model simplicity ordering corrected:
                          logreg(0) < DT(1) < RF(2) < XGB(3) < LGB(4) <
                          CatBoost(5) < SVM(6) < MLP(7).
"""

import warnings
import numpy as np
from scipy.stats import loguniform

from sklearn.base import clone
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import (
    StratifiedKFold, StratifiedGroupKFold, RandomizedSearchCV, cross_val_predict,
)
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score, matthews_corrcoef,
    recall_score, balanced_accuracy_score, brier_score_loss, confusion_matrix,
)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTENC, RandomOverSampler

import config

warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", message="X does not have valid feature names.*")


def _as_lgb_numpy(X):
    """Return a plain numeric matrix for LightGBM inside sklearn pipelines."""
    if hasattr(X, "to_numpy"):
        return X.to_numpy()
    if hasattr(X, "toarray"):
        return X.toarray()
    return np.asarray(X)


class QuietLGBMClassifier(LGBMClassifier):
    """LightGBM wrapper that uses the same matrix type for fit and predict."""

    def fit(self, X, y, **kwargs):
        return super().fit(_as_lgb_numpy(X), y, **kwargs)

    def predict(self, X, **kwargs):
        return super().predict(_as_lgb_numpy(X), **kwargs)

    def predict_proba(self, X, **kwargs):
        return super().predict_proba(_as_lgb_numpy(X), **kwargs)


# --------------------------------------------------------------------------- #
# Estimators
# --------------------------------------------------------------------------- #
def make_classifier(name, imbalance, meta, pos_weight, seed):
    """Return a fresh estimator with the requested imbalance handling.

    imbalance in {"none", "balanced", "smotenc"}.  For "smotenc" the resampler
    lives in the pipeline, so the estimator itself carries no class weighting.

    FIX 2: LightGBM now sets is_unbalance=True when imbalance="balanced".
    Previously it was the only model that silently ignored the balanced flag.

    FIX 3: CatBoost now sets thread_count=1 for cross-machine reproducibility.
    """
    weighted = imbalance == "balanced"

    if name == "logreg":
        return LogisticRegression(
            penalty="l2", solver="lbfgs", max_iter=4000,
            class_weight="balanced" if weighted else None, random_state=seed,
        )
    if name == "decision_tree":
        return DecisionTreeClassifier(
            class_weight="balanced" if weighted else None, random_state=seed,
        )
    if name == "random_forest":
        return RandomForestClassifier(
            n_estimators=400, n_jobs=config.N_JOBS,
            class_weight="balanced" if weighted else None, random_state=seed,
        )
    if name == "svm":
        return SVC(
            kernel="rbf", probability=True,
            class_weight="balanced" if weighted else None, random_state=seed,
        )
    if name == "mlp":
        # No class weighting available; imbalance handled by the pipeline sampler.
        return MLPClassifier(
            max_iter=600, early_stopping=True, n_iter_no_change=15,
            random_state=seed,
        )
    if name == "xgboost":
        return XGBClassifier(
            n_estimators=400, tree_method="hist", eval_metric="logloss",
            scale_pos_weight=pos_weight if weighted else 1.0,
            n_jobs=config.N_JOBS, random_state=seed, verbosity=0,
        )
    if name == "lightgbm":
        # FIX 2: is_unbalance=True applies class-weight correction inside
        # LightGBM when imbalance="balanced", consistent with all other models.
        # n_jobs=1 was already correct here; kept unchanged.
        return QuietLGBMClassifier(
            verbose=-1,
            verbosity=-1,
            is_unbalance=weighted,   # FIX 2: was omitted entirely
            random_state=seed,
            n_jobs=1,
        )
    if name == "catboost":
        # FIX 3: thread_count=1 ensures identical results on machines with
        # different CPU core counts (CatBoost otherwise uses all cores).
        # All categoricals here are binary/ordinal integer codes, so they are
        # handled as numeric splits.
        return CatBoostClassifier(
            iterations=400, random_seed=seed, verbose=0,
            auto_class_weights="Balanced" if weighted else None,
            thread_count=1,          # FIX 3: was unset (= all cores)
        )
    raise ValueError(f"unknown model {name}")


def param_space(name):
    """Randomized-search distributions, keyed by the 'clf__' pipeline step."""
    if name == "logreg":
        return {"clf__C": loguniform(1e-3, 1e2)}
    if name == "decision_tree":
        return {
            "clf__max_depth": [2, 3, 4, 6, 8, 12, None],
            "clf__min_samples_leaf": [1, 2, 5, 10, 20],
            "clf__min_samples_split": [2, 5, 10, 20],
            "clf__ccp_alpha": [0.0, 1e-4, 1e-3, 1e-2],
        }
    if name == "random_forest":
        return {
            "clf__n_estimators": [200, 400, 600],
            "clf__max_depth": [None, 6, 10, 16],
            "clf__max_features": ["sqrt", "log2", 0.5],
            "clf__min_samples_leaf": [1, 2, 5],
        }
    if name == "xgboost":
        return {
            "clf__n_estimators": [200, 400, 600],
            "clf__max_depth": [3, 4, 6, 8],
            "clf__learning_rate": loguniform(1e-2, 3e-1),
            "clf__subsample": [0.7, 0.85, 1.0],
            "clf__colsample_bytree": [0.7, 0.85, 1.0],
            "clf__min_child_weight": [1, 3, 5],
            "clf__reg_lambda": [0.5, 1.0, 2.0, 5.0],
        }
    if name == "lightgbm":
        return {
            "clf__n_estimators": [200, 400, 600],
            "clf__num_leaves": [15, 31, 63],
            "clf__max_depth": [-1, 4, 8],
        }
    if name == "catboost":
        return {
            "clf__depth": [4, 6, 8],
            "clf__learning_rate": [round(x, 6) for x in np.linspace(0.01, 0.30, 30).tolist()],
            "clf__iterations": [200, 400, 600],
            "clf__l2_leaf_reg": [1, 3, 5, 9],
        }
    if name == "svm":
        return {
            "clf__C": loguniform(1e-2, 1e2),
            "clf__gamma": ["scale", 1e-3, 1e-2, 1e-1],
        }
    if name == "mlp":
        return {
            "clf__hidden_layer_sizes": [(64,), (128,), (64, 32), (128, 64)],
            "clf__alpha": loguniform(1e-5, 1e-2),
            "clf__learning_rate_init": [1e-3, 5e-4],
            "clf__batch_size": [32, 64, 128],
        }
    raise ValueError(name)


def build_pipeline(name, imbalance, meta, pos_weight, seed):
    """Assemble imputation, optional scaling, optional sampler, and estimator."""
    num_steps = [("imputer", SimpleImputer(strategy="median"))]
    if name in config.SCALED_MODELS:
        num_steps.append(("scaler", StandardScaler()))

    cat_steps = [("imputer", SimpleImputer(strategy="most_frequent"))]
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", SkPipeline(num_steps), meta.cont_idx),
            ("cat", SkPipeline(cat_steps), meta.cat_idx),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )

    steps = [("preprocess", preprocessor)]
    transformed_cat_idx = list(range(len(meta.cont_idx), len(meta.cont_idx) + len(meta.cat_idx)))

    if imbalance == "smotenc" and transformed_cat_idx:
        steps.append(("sampler", SMOTENC(categorical_features=transformed_cat_idx,
                                         random_state=seed)))
    elif imbalance == "balanced" and name == "mlp":
        steps.append(("sampler", RandomOverSampler(random_state=seed)))

    steps.append(("clf", make_classifier(name, imbalance, meta, pos_weight, seed)))
    return ImbPipeline(steps)


# --------------------------------------------------------------------------- #
# Metrics
# --------------------------------------------------------------------------- #
def _specificity(y, pred):
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return tn / (tn + fp) if (tn + fp) else np.nan


def _safe_roc(y, p, d):
    try:
        return roc_auc_score(y, p) if len(np.unique(y)) >= 2 else np.nan
    except Exception:
        return np.nan


def _safe_pr(y, p, d):
    try:
        return average_precision_score(y, p) if len(np.unique(y)) >= 2 else np.nan
    except Exception:
        return np.nan


def compute_metrics(y, proba, pred):
    return {
        "ROC_AUC": _safe_roc(y, proba, pred),
        "PR_AUC": _safe_pr(y, proba, pred),
        "F1": f1_score(y, pred, zero_division=0),
        "MCC": matthews_corrcoef(y, pred),
        "Sensitivity": recall_score(y, pred, zero_division=0),
        "Specificity": _specificity(y, pred),
        "Balanced_Acc": balanced_accuracy_score(y, pred),
        "Brier": brier_score_loss(y, proba),
    }


METRIC_FUNCS = {
    "ROC_AUC": _safe_roc,
    "PR_AUC": _safe_pr,
    "F1": lambda y, p, d: f1_score(y, d, zero_division=0),
    "MCC": lambda y, p, d: matthews_corrcoef(y, d),
    "Sensitivity": lambda y, p, d: recall_score(y, d, zero_division=0),
    "Specificity": lambda y, p, d: _specificity(y, d),
    "Balanced_Acc": lambda y, p, d: balanced_accuracy_score(y, d),
    "Brier": lambda y, p, d: brier_score_loss(y, p),
}


def _native_params(d):
    """Cast numpy scalar values to native python types.

    Guards against a known CatBoostClassifier/sklearn incompatibility: if a
    numpy.floating/integer hyperparameter (e.g. sampled by RandomizedSearchCV)
    is fed into CatBoostClassifier, sklearn.base.clone() can raise
    RuntimeError("Cannot clone object ..., as the constructor either does not
    set or modifies parameter ...") because CatBoost's get_params() returns a
    plain python float/int that fails clone's identity check against the
    original numpy scalar. Casting up front avoids the mismatch.
    """
    out = {}
    for k, v in d.items():
        if isinstance(v, np.floating):
            out[k] = float(v)
        elif isinstance(v, np.integer):
            out[k] = int(v)
        else:
            out[k] = v
    return out


def best_mcc_threshold(y, proba):
    """Decision threshold maximizing MCC (scanned over candidate cut-points)."""
    grid = np.unique(np.concatenate([[0.0, 1.0], np.quantile(proba, np.linspace(0, 1, 101))]))
    best_t, best_m = 0.5, -2.0
    for t in grid:
        pred = (proba >= t).astype(int)
        if pred.sum() == 0 or pred.sum() == len(pred):
            continue
        m = matthews_corrcoef(y, pred)
        if m > best_m:
            best_m, best_t = m, t
    return float(best_t)


# --------------------------------------------------------------------------- #
# Cross-validation splitters
# --------------------------------------------------------------------------- #
def outer_splitter(meta, seed):
    if meta.grouped:
        return StratifiedGroupKFold(n_splits=config.OUTER_SPLITS, shuffle=True,
                                    random_state=seed)
    return StratifiedKFold(n_splits=config.OUTER_SPLITS, shuffle=True,
                           random_state=seed)


def inner_splitter(meta, seed):
    if meta.grouped:
        return StratifiedGroupKFold(n_splits=config.INNER_SPLITS, shuffle=True,
                                    random_state=seed + 1000)
    return StratifiedKFold(n_splits=config.INNER_SPLITS, shuffle=True,
                           random_state=seed + 1000)


def _pos_weight(y):
    pos = max(int(y.sum()), 1)
    neg = len(y) - int(y.sum())
    return neg / pos


# --------------------------------------------------------------------------- #
# Nested CV for one (model, dataset, seed)
# --------------------------------------------------------------------------- #
def run_nested_cv(meta, name, seed, imbalance="balanced", n_iter=None,
                  keep_estimators=False):
    """Return OOF probabilities/labels/fold-ids and per-fold metric dicts.

    If keep_estimators is True, also return the fitted pipeline and test indices
    for every outer fold (used by the explainability analysis).

    FIX 1: The decision threshold is now computed AFTER the inner loop has
    filled inner_proba for every validation sample.  Previously the call to
    best_mcc_threshold was inside the loop body, where inner_proba still
    contained NaN for unseen splits.  np.quantile propagates NaN, so every
    intermediate threshold was invalid.
    """
    X, y, groups = meta.X, meta.y, meta.groups
    n = len(y)
    oof_proba = np.full(n, np.nan)
    oof_pred = np.full(n, -1, dtype=int)
    fold_id = np.full(n, -1, dtype=int)
    fold_metrics, fold_params = [], []
    fold_summaries = []
    fold_estimators, fold_test_idx = [], []

    scoring = meta.primary
    if n_iter is None:
        n_iter = config.N_ITER_BOOST if name in {"xgboost", "lightgbm", "catboost"} \
            else config.N_ITER_SIMPLE

    osplit = outer_splitter(meta, seed)
    split_args = (X, y, groups) if meta.grouped else (X, y)

    for k, (tr, te) in enumerate(osplit.split(*split_args)):
        Xtr, Xte = X.iloc[tr], X.iloc[te]
        ytr, yte = y[tr], y[te]
        gtr = groups[tr] if meta.grouped else None
        pw = _pos_weight(ytr)

        pipe = build_pipeline(name, imbalance, meta, pw, seed)
        inner = inner_splitter(meta, seed)
        search = RandomizedSearchCV(
            pipe, param_space(name), n_iter=n_iter, scoring=scoring,
            cv=inner, random_state=seed, refit=True, n_jobs=config.N_JOBS,
            error_score="raise",
        )
        if meta.grouped:
            search.fit(Xtr, ytr, groups=gtr)
        else:
            search.fit(Xtr, ytr)

        # ------------------------------------------------------------------ #
        # FIX 1: Compute threshold on FULLY-POPULATED inner OOF probabilities.
        # Collect all inner-fold predictions first, then call best_mcc_threshold
        # once the array has no NaNs.  Previously this call was inside the loop.
        # ------------------------------------------------------------------ #
        inner_proba = np.full(len(ytr), np.nan)
        for tr2, val2 in inner_splitter(meta, seed).split(Xtr, ytr, gtr):
            fold_pipe = build_pipeline(name, imbalance, meta, pw, seed)
            fold_pipe.set_params(**_native_params(search.best_params_))
            fold_pipe.fit(Xtr.iloc[tr2], ytr[tr2])
            inner_proba[val2] = fold_pipe.predict_proba(Xtr.iloc[val2])[:, 1]
        # threshold computed here — inner_proba is fully populated (no NaNs)
        thr = best_mcc_threshold(ytr, inner_proba)

        proba = search.best_estimator_.predict_proba(Xte)[:, 1]
        pred = (proba >= thr).astype(int)

        oof_proba[te] = proba
        oof_pred[te] = pred
        fold_id[te] = k
        m = compute_metrics(yte, proba, pred)
        m["threshold"] = thr
        fold_metrics.append(m)
        best_params = _native_params(search.best_params_)
        fold_params.append(best_params)
        tn, fp, fn, tp = confusion_matrix(yte, pred, labels=[0, 1]).ravel()
        fold_summaries.append({
            "fold": int(k),
            "train_n": int(len(tr)),
            "test_n": int(len(te)),
            "train_positive": int(np.sum(ytr == 1)),
            "train_negative": int(np.sum(ytr == 0)),
            "test_positive": int(np.sum(yte == 1)),
            "test_negative": int(np.sum(yte == 0)),
            "train_groups": int(len(np.unique(gtr))) if meta.grouped else None,
            "test_groups": int(len(np.unique(groups[te]))) if meta.grouped else None,
            "threshold": float(thr),
            "best_params": best_params,
            "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
            "train_indices": [int(i) for i in tr],
            "test_indices": [int(i) for i in te],
            **{kk: float(vv) if vv is not None and np.isfinite(vv) else np.nan
               for kk, vv in m.items()},
        })
        if keep_estimators:
            fold_estimators.append(search.best_estimator_)
            fold_test_idx.append(te)

    return {
        "oof_proba": oof_proba, "oof_pred": oof_pred, "fold_id": fold_id,
        "fold_metrics": fold_metrics, "fold_params": fold_params,
        "fold_summaries": fold_summaries,
        "fold_estimators": fold_estimators, "fold_test_idx": fold_test_idx,
    }


# --------------------------------------------------------------------------- #
# Bootstrap confidence intervals (patient-level resampling when grouped)
# --------------------------------------------------------------------------- #
def bootstrap_ci(y, proba, pred, groups, metric_name, n_boot=None, seed=None):
    """Compute 95% bootstrap CI for a metric.

    FIX 4 (reproducibility): accepts an explicit seed so each call uses
    independent random draws.  Callers should pass a unique seed per call
    (e.g. based on dataset index and comparison tag) rather than always
    defaulting to config.RNG=42, which would recycle identical draws.
    """
    n_boot = n_boot or config.N_BOOTSTRAP
    seed = config.RNG if seed is None else seed
    rs = np.random.RandomState(seed)
    func = METRIC_FUNCS[metric_name]
    vals = []

    if groups is not None:
        uniq = np.unique(groups)
        idx_by_group = {g: np.where(groups == g)[0] for g in uniq}
        for _ in range(n_boot):
            chosen = rs.choice(uniq, size=len(uniq), replace=True)
            idx = np.concatenate([idx_by_group[g] for g in chosen])
            if len(np.unique(y[idx])) < 2:
                continue
            vals.append(func(y[idx], proba[idx], pred[idx]))
    else:
        n = len(y)
        for _ in range(n_boot):
            idx = rs.randint(0, n, n)
            if len(np.unique(y[idx])) < 2:
                continue
            vals.append(func(y[idx], proba[idx], pred[idx]))

    vals = np.array(vals)
    if vals.size == 0:
        return np.nan, np.nan
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))


# --------------------------------------------------------------------------- #
# Calibration metrics
# --------------------------------------------------------------------------- #
def calibration_metrics(y, proba):
    if len(np.unique(y)) < 2:
        return {"Brier": float(brier_score_loss(y, proba)),
                "Calibration_slope": np.nan,
                "Calibration_intercept": np.nan}
    eps = 1e-6
    p = np.clip(proba, eps, 1 - eps)
    logit = np.log(p / (1 - p))
    lr = LogisticRegression(solver="lbfgs", max_iter=1000)
    lr.fit(logit.reshape(-1, 1), y)
    slope = float(lr.coef_[0, 0])
    intercept = float(lr.intercept_[0])
    return {
        "Brier": float(brier_score_loss(y, proba)),
        "Calibration_slope": slope,
        "Calibration_intercept": intercept,
    }


# --------------------------------------------------------------------------- #
# Model selection rule
# --------------------------------------------------------------------------- #
def select_model(meta, agg_table):
    """Pick one final model per dataset, plus a tree model for SHAP.

    Order: primary metric (mean), then fold stability (std), then calibration
    (Brier), then simplicity. Models within one std of the best primary mean are
    treated as tied; among ties a tree model is preferred for explainability.

    FIX 5: Corrected simplicity ordering to the conventional ML explainability
    preference: logreg(0) < DT(1) < RF(2) < XGB(3) < LGB(4) < CatBoost(5)
    < SVM(6) < MLP(7).  The original placed CatBoost at 2 and RF at 5, which
    would prefer a CatBoost ensemble over a Random Forest — the reverse of the
    standard recommendation.
    """
    primary_col = "PR_AUC" if meta.primary == "average_precision" else "ROC_AUC"
    means = agg_table[(primary_col, "mean")].astype(float)
    stds = agg_table[(primary_col, "std")].astype(float).fillna(0.0)
    if means.notna().any():
        best = means.idxmax()
    else:
        best = means.index[0]
        means = means.fillna(-np.inf)
    tied = means[means >= means[best] - stds[best]].index.tolist()

    # FIX 5: conventional explainability simplicity order
    simplicity = {
        "logreg": 0, "decision_tree": 1, "random_forest": 2,
        "xgboost": 3, "lightgbm": 4, "catboost": 5,
        "svm": 6, "mlp": 7,
    }

    def key(m):
        return (
            -means[m],
            agg_table[("Brier", "mean")][m],
            stds[m],
            simplicity.get(m, 9),
        )

    selected = sorted(tied, key=key)[0]

    tree_tied = [m for m in tied if m in config.TREE_MODELS]
    if selected in config.TREE_MODELS:
        selected_tree = selected
    elif tree_tied:
        selected_tree = sorted(tree_tied, key=key)[0]
    else:
        all_trees = [m for m in means.index if m in config.TREE_MODELS]
        selected_tree = means[all_trees].idxmax()

    return selected, selected_tree

Writing modeling.py


## 3. Explainability

In [6]:
%%writefile explain.py
"""
Explainability analysis for the selected model of the active dataset.

Components:
  1. Global SHAP computed per outer fold, then aggregated into a stability table
     (mean |SHAP|, std, top-10 frequency, median rank).
  2. Permutation importance on outer test data, aggregated across folds.
  3. SHAP vs permutation agreement (Spearman rank correlation, top-10 overlap).
  4. PDP + ICE for the top stable continuous predictors.
  5. Local SHAP waterfall plots for a representative true positive and false
     negative for the active dataset.
  6. LIME cross-check on those same local cases.

TreeSHAP is applied to a tree model. For visualization (PDP/ICE, local cases,
visualization) a single tuned model is fit on the full dataset; case *selection*
uses honest out-of-fold predictions.
"""

import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

import shap
try:
    from lime.lime_tabular import LimeTabularExplainer
except Exception:
    LimeTabularExplainer = None
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.model_selection import RandomizedSearchCV

import config
import modeling as M

warnings.filterwarnings("ignore")


# --------------------------------------------------------------------------- #
def _clf_of(estimator):
    return estimator.named_steps["clf"]


def _model_matrix(estimator, X):
    """Return the matrix actually seen by the final estimator."""
    if hasattr(estimator, "named_steps") and "preprocess" in estimator.named_steps:
        Xt = estimator.named_steps["preprocess"].transform(X)
    else:
        Xt = X
    if hasattr(Xt, "toarray"):
        Xt = Xt.toarray()
    return np.asarray(Xt, dtype=float)


def _shap_values_pos(clf, X):
    """Return (n_samples, n_features) SHAP values for the positive class."""
    explainer = shap.TreeExplainer(clf)
    sv = explainer.shap_values(X)
    sv = np.asarray(sv)
    if sv.ndim == 3:
        if sv.shape[0] == 2:        # (classes, n, f)
            sv = sv[1]
        elif sv.shape[-1] == 2:     # (n, f, classes)
            sv = sv[..., 1]
        else:
            sv = sv[..., -1]
    return sv


def fit_final(meta, name, seed=config.REPORT_SEED, imbalance="balanced", return_metadata=False):
    """Fit a tuned model on the entire dataset (for visualization only)."""
    pw = M._pos_weight(meta.y)
    pipe = M.build_pipeline(name, imbalance, meta, pw, seed)
    n_iter = config.N_ITER_BOOST if name in {"xgboost", "lightgbm", "catboost"} \
        else config.N_ITER_SIMPLE
    search = RandomizedSearchCV(
        pipe, M.param_space(name), n_iter=n_iter, scoring=meta.primary,
        cv=M.inner_splitter(meta, seed), random_state=seed, refit=True, n_jobs=config.N_JOBS,
    )
    if meta.grouped:
        search.fit(meta.X, meta.y, groups=meta.groups)
    else:
        search.fit(meta.X, meta.y)
    final_model = search.best_estimator_
    metadata = {
        "dataset": meta.name,
        "model": name,
        "seed": int(seed),
        "imbalance": imbalance,
        "primary_metric": meta.primary,
        "n_iter": int(n_iter),
        "best_score": float(search.best_score_),
        "best_params": M._native_params(search.best_params_),
        "search_space": {k: repr(v) for k, v in M.param_space(name).items()},
    }
    final_model._ckd_final_fit_metadata = metadata
    if return_metadata:
        return final_model, metadata
    return final_model


# --------------------------------------------------------------------------- #
# 1. Global SHAP stability across outer folds
# --------------------------------------------------------------------------- #
def shap_stability(meta, tree_name, fold_estimators, fold_test_idx):
    feats = list(meta.X.columns)
    per_fold_mean_abs, per_fold_rank, per_fold_signed = [], [], []

    last_sv, last_X = None, None
    for est, te in zip(fold_estimators, fold_test_idx):
        Xte = meta.X.iloc[te]
        Xte_model = _model_matrix(est, Xte)
        sv = _shap_values_pos(_clf_of(est), Xte_model)
        mean_abs = np.abs(sv).mean(axis=0)
        signed = sv.mean(axis=0)
        order = pd.Series(mean_abs, index=feats).rank(ascending=False)
        per_fold_mean_abs.append(mean_abs)
        per_fold_signed.append(signed)
        per_fold_rank.append(order.values)
        last_sv, last_X = sv, Xte_model

    mean_abs = np.vstack(per_fold_mean_abs)
    ranks = np.vstack(per_fold_rank)
    signed = np.vstack(per_fold_signed)

    summary = pd.DataFrame({
        "feature": feats,
        "mean_abs_shap": mean_abs.mean(axis=0),
        "std_abs_shap": mean_abs.std(axis=0),
        "median_rank": np.median(ranks, axis=0),
        "top10_frequency": (ranks <= 10).mean(axis=0),
        "mean_signed_shap": signed.mean(axis=0),
    }).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
    summary["category"] = summary["feature"].map(meta.feature_categories)

    # Beeswarm from the last fold's test set (representative single view).
    plt.figure()
    expl = shap.Explanation(values=last_sv, data=last_X,
                            feature_names=feats)
    shap.plots.beeswarm(expl, max_display=15, show=False)
    plt.title(f"SHAP summary - {meta.name}")
    plt.tight_layout()
    beeswarm_path = config.FIGDIR / f"shap_beeswarm_{meta.name}.png"
    plt.savefig(beeswarm_path, dpi=config.DPI, bbox_inches="tight")
    plt.close()

    return summary, str(beeswarm_path)


# --------------------------------------------------------------------------- #
# 2. Permutation importance across outer folds
# --------------------------------------------------------------------------- #
def permutation_summary(meta, fold_estimators, fold_test_idx, seed=config.REPORT_SEED):
    feats = list(meta.X.columns)
    scoring = "roc_auc" if meta.primary == "roc_auc" else "average_precision"
    mat = []
    for est, te in zip(fold_estimators, fold_test_idx):
        Xte, yte = meta.X.iloc[te], meta.y[te]
        r = permutation_importance(est, Xte, yte, scoring=scoring,
                                   n_repeats=config.PERM_REPEATS,
                                   random_state=seed, n_jobs=config.N_JOBS)
        mat.append(r.importances_mean)
    mat = np.vstack(mat)
    summary = pd.DataFrame({
        "feature": feats,
        "perm_importance": mat.mean(axis=0),
        "perm_std": mat.std(axis=0),
    }).sort_values("perm_importance", ascending=False).reset_index(drop=True)
    summary["perm_rank"] = np.arange(1, len(summary) + 1)
    return summary


def shap_perm_agreement(shap_summary, perm_summary, k=10):
    merged = shap_summary.merge(perm_summary, on="feature")
    shap_rank = merged["mean_abs_shap"].rank(ascending=False)
    perm_rank = merged["perm_importance"].rank(ascending=False)
    rho, _ = spearmanr(shap_rank, perm_rank)
    shap_top = set(shap_summary["feature"].head(k))
    perm_top = set(perm_summary["feature"].head(k))
    overlap = len(shap_top & perm_top) / k
    return {"spearman_rho": float(rho), f"top{k}_overlap": float(overlap)}


# --------------------------------------------------------------------------- #
# 4. PDP + ICE for top stable continuous features
# --------------------------------------------------------------------------- #
def pdp_ice(meta, final_model, shap_summary, max_panels=2):
    cont = set(meta.continuous)
    top_cont = [f for f in shap_summary["feature"] if f in cont][:max_panels]
    if not top_cont:
        return None, []
    fig, axes = plt.subplots(1, len(top_cont), figsize=(6 * len(top_cont), 4.5),
                             squeeze=False)
    for ax, feat in zip(axes[0], top_cont):
        PartialDependenceDisplay.from_estimator(
            final_model, meta.X, [feat], kind="both", ax=ax,
            ice_lines_kw={"alpha": 0.15, "color": "tab:blue"},
            pd_line_kw={"color": "black", "linewidth": 2.5},
        )
        ax.set_title(f"{feat}")
    fig.suptitle(f"PDP + ICE - {meta.name}")
    fig.tight_layout()
    path = config.FIGDIR / f"pdp_ice_{meta.name}.png"
    fig.savefig(path, dpi=config.DPI, bbox_inches="tight")
    plt.close(fig)
    return str(path), top_cont


# --------------------------------------------------------------------------- #
# 5 + 6. Local SHAP waterfalls and LIME cross-check
# --------------------------------------------------------------------------- #
def _pick_cases(meta, oof_proba, oof_pred):
    """Representative true positive and false negative (median probability)."""
    y = meta.y
    tp = np.where((y == 1) & (oof_pred == 1))[0]
    fn = np.where((y == 1) & (oof_pred == 0))[0]
    out = {}
    for label, idxs in [("TP", tp), ("FN", fn)]:
        if len(idxs) == 0:
            continue
        probs = oof_proba[idxs]
        out[label] = int(idxs[np.argsort(probs)[len(probs) // 2]])
    return out


def local_explanations(meta, final_model, oof_proba, oof_pred):
    feats = list(meta.X.columns)
    clf = _clf_of(final_model)
    cases = _pick_cases(meta, oof_proba, oof_pred)
    explainer = shap.TreeExplainer(clf)

    base = explainer.expected_value
    if isinstance(base, (list, np.ndarray)):
        base = np.asarray(base).ravel()
        base = float(base[-1] if base.size > 1 else base[0])

    lime_exp = None
    if LimeTabularExplainer is not None:
        lime_exp = LimeTabularExplainer(
            training_data=meta.X.values, feature_names=feats,
            class_names=["non-CKD", "CKD"], categorical_features=meta.cat_idx,
            discretize_continuous=True, random_state=config.RNG, mode="classification",
        )

    records, paths = [], {}
    for label, idx in cases.items():
        x_row = meta.X.iloc[[idx]]
        x_row_model = _model_matrix(final_model, x_row)
        sv = _shap_values_pos(clf, x_row_model)[0]

        # SHAP waterfall
        plt.figure()
        expl = shap.Explanation(values=sv, base_values=base,
                                data=meta.X.iloc[idx].values, feature_names=feats)
        shap.plots.waterfall(expl, max_display=12, show=False)
        plt.title(f"{meta.name} - {label} (p={oof_proba[idx]:.2f})")
        plt.tight_layout()
        p = config.FIGDIR / f"local_shap_{meta.name}_{label}.png"
        plt.savefig(p, dpi=config.DPI, bbox_inches="tight")
        plt.close()
        paths[f"shap_{label}"] = str(p)

        # LIME on the same instance.
        lime_pairs = []
        agreement = np.nan
        if lime_exp is not None:
            lex = lime_exp.explain_instance(
                meta.X.iloc[idx].values, final_model.predict_proba,
                num_features=10, labels=(1,))
            lime_pairs = lex.as_list(label=1)
            fig = lex.as_pyplot_figure(label=1)
            fig.suptitle(f"LIME {meta.name} - {label}")
            fig.tight_layout()
            pl = config.FIGDIR / f"local_lime_{meta.name}_{label}.png"
            fig.savefig(pl, dpi=config.DPI, bbox_inches="tight")
            plt.close(fig)
            paths[f"lime_{label}"] = str(pl)

            shap_sign = {f: np.sign(s) for f, s in zip(feats, sv)}
            agree = []
            for desc, weight in lime_pairs:
                base_feat = next((f for f in feats if f in desc), None)
                if base_feat is not None:
                    agree.append(np.sign(weight) == shap_sign.get(base_feat, 0))
            agreement = float(np.mean(agree)) if agree else np.nan

        records.append({
            "dataset": meta.name, "case": label, "row_index": idx,
            "true_class": int(meta.y[idx]), "pred_class": int(oof_pred[idx]),
            "pred_proba": float(oof_proba[idx]),
            "top_shap_support": ", ".join(
                pd.Series(sv, index=feats).sort_values(ascending=False).head(3).index),
            "top_shap_oppose": ", ".join(
                pd.Series(sv, index=feats).sort_values().head(3).index),
            "shap_lime_sign_agreement": agreement,
        })
    return pd.DataFrame(records), paths


Writing explain.py


## 4. Reporting, Tables and Figures

In [7]:
%%writefile report.py
"""
Tables and figures for one CKD dataset.

The notebook writes dataset characteristics, descriptive statistics, model
performance, robustness results, explanation summaries, ROC and precision recall
curves, confusion and calibration plots, workflow, SHAP, PDP and ICE, local
explanations, and learning curves.
"""

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

from sklearn.metrics import roc_curve, precision_recall_curve, confusion_matrix
from sklearn.calibration import calibration_curve
from sklearn.model_selection import learning_curve, StratifiedKFold, StratifiedGroupKFold

import config

METRIC_ORDER = ["ROC_AUC", "PR_AUC", "F1", "MCC", "Sensitivity",
                "Specificity", "Balanced_Acc", "Brier"]


def _save_table(df, name, float_fmt="%.3f"):
    config.TABDIR.mkdir(parents=True, exist_ok=True)
    df.to_csv(config.TABDIR / f"{name}.csv")
    try:
        df.to_latex(config.TABDIR / f"{name}.tex", float_format=float_fmt,
                    escape=False)
    except Exception:
        pass


def aggregate_metrics(model_fold_metrics):
    rows = {}
    for model, folds in model_fold_metrics.items():
        rows[model] = pd.DataFrame(folds)[METRIC_ORDER]
    agg = {}
    for model, df in rows.items():
        agg[model] = {}
        for met in METRIC_ORDER:
            agg[model][(met, "mean")] = df[met].mean()
            agg[model][(met, "std")] = df[met].std()
    table = pd.DataFrame(agg).T
    table.columns = pd.MultiIndex.from_tuples(table.columns)
    return table


def performance_table(agg, primary, selected, fname):
    primary_metric = "PR_AUC" if primary == "average_precision" else "ROC_AUC"
    cols = [primary_metric] + [m for m in METRIC_ORDER if m != primary_metric]
    disp = pd.DataFrame(index=[config.MODEL_LABELS.get(m, m) for m in agg.index])
    for met in cols:
        means = agg[(met, "mean")].values
        stds = agg[(met, "std")].values
        disp[met] = [f"{m:.3f} ± {s:.3f}" for m, s in zip(means, stds)]
    disp.index.name = "Model"
    disp.attrs["selected"] = config.MODEL_LABELS.get(selected, selected)
    _save_table(disp, fname, float_fmt=None)
    return disp


def table1_characteristics(meta):
    y = meta.y
    n_pos = int(y.sum())
    n_neg = int(len(y) - n_pos)
    n_patients = len(np.unique(meta.groups)) if meta.groups is not None else len(y)
    rec_per_patient = len(y) / n_patients if meta.groups is not None else 1
    block = {
        "Patients": n_patients,
        "Records": len(y),
        "Records per patient": round(rec_per_patient, 1),
        "Predictors": meta.X.shape[1],
        "Continuous predictors": len(meta.continuous),
        "Categorical predictors": len(meta.categorical),
        "CKD positive (n, % )": f"{n_pos} ({100*n_pos/len(y):.1f}%)",
        "CKD negative (n, % )": f"{n_neg} ({100*n_neg/len(y):.1f}%)",
        "Missing values": int(meta.X.isna().sum().sum()),
        "Primary metric": "PR-AUC" if meta.primary == "average_precision" else "ROC-AUC",
        "Validation": config.VALIDATION_SUMMARY,
        "Removed predictors": ", ".join(config.REMOVED_PREDICTORS) if config.REMOVED_PREDICTORS else "None",
        "Derived predictors": ", ".join(config.DERIVED_PREDICTORS) if config.DERIVED_PREDICTORS else "None",
    }
    df = pd.DataFrame({config.DATASET_LABEL: block})
    _save_table(df, "table1_dataset_characteristics", float_fmt=None)
    return df


def descriptive_stats(meta, fname):
    rows = {}
    for c in meta.continuous:
        s = meta.X[c]
        rows[c] = {"type": "continuous", "mean": s.mean(), "std": s.std(),
                   "median": s.median(),
                   "IQR": f"{s.quantile(.25):.2f} to {s.quantile(.75):.2f}"}
    for c in meta.categorical:
        s = meta.X[c]
        vc = s.value_counts(normalize=True)
        top = vc.index[0]
        rows[c] = {"type": "categorical", "mode": top,
                   "mode_pct": f"{100*vc.iloc[0]:.1f}%", "n_levels": s.nunique()}
    df = pd.DataFrame(rows).T
    _save_table(df, fname, float_fmt="%.3f")
    return df


def table5_explanations(shap_summary, perm_summary, agreement, fname):
    merged = shap_summary.merge(
        perm_summary[["feature", "perm_rank", "perm_importance"]], on="feature")
    merged["shap_rank"] = merged["mean_abs_shap"].rank(ascending=False).astype(int)
    out = merged.head(10)[[
        "feature", "category", "mean_abs_shap", "shap_rank",
        "median_rank", "top10_frequency", "perm_rank", "mean_signed_shap",
    ]].copy()
    out = out.rename(columns={
        "mean_abs_shap": "mean|SHAP|", "median_rank": "SHAP median rank",
        "top10_frequency": "top-10 freq", "perm_rank": "permutation rank",
        "mean_signed_shap": "mean signed SHAP",
    })
    out.attrs["agreement"] = agreement
    _save_table(out.set_index("feature"), fname, float_fmt="%.4f")
    return out


def figure1_workflow(meta):
    n_patients = len(np.unique(meta.groups)) if meta.groups is not None else len(meta.y)
    n_pos = int(meta.y.sum())
    n_neg = int(len(meta.y) - n_pos)
    if meta.groups is not None:
        dataset_text = (f"{config.DATASET_TITLE}\n{n_patients} patients, {len(meta.y):,} records\n"
                        f"{meta.X.shape[1]} predictors, {n_pos} CKD and {n_neg} non CKD records")
    else:
        dataset_text = (f"{config.DATASET_TITLE}\n{n_patients} patients, {meta.X.shape[1]} predictors\n"
                        f"{n_pos} CKD and {n_neg} non CKD")

    fig, ax = plt.subplots(figsize=(10, 8))
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 10)
    ax.axis("off")

    def box(x, y, w, h, text, color):
        ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.08",
                                    linewidth=1.4, edgecolor="#333", facecolor=color))
        ax.text(x + w / 2, y + h / 2, text, ha="center", va="center",
                fontsize=9, wrap=True)

    def arrow(x1, y1, x2, y2):
        ax.add_patch(FancyArrowPatch((x1, y1), (x2, y2),
                     arrowstyle="-|>", mutation_scale=14, color="#555"))

    box(1.3, 8.4, 7.4, 1.1, dataset_text, "#dbeafe")
    box(1.3, 6.8, 7.4, 1.1, config.PREPROCESSING_SUMMARY, "#dcfce7")
    box(1.3, 5.2, 7.4, 1.0, config.VALIDATION_SUMMARY, "#fef9c3")
    box(1.3, 3.6, 7.4, 1.0, config.MODEL_SUMMARY, "#fee2e2")
    box(1.3, 2.0, 7.4, 1.0,
        "Evaluation: ROC AUC, PR AUC, F1, MCC, sensitivity, specificity, balanced accuracy, Brier, threshold by MCC, bootstrap confidence intervals",
        "#fae8ff")
    box(1.3, 0.4, 7.4, 1.0, config.XAI_SUMMARY, "#cffafe")

    for y1, y2 in [(8.4, 7.9), (6.8, 6.3), (5.2, 4.7), (3.6, 3.1), (2.0, 1.5)]:
        arrow(5.0, y1, 5.0, y2)
    ax.set_title("Study workflow", fontsize=13)
    fig.tight_layout()
    for ext in ("png", "pdf"):
        fig.savefig(config.FIGDIR / f"figure1_workflow.{ext}", dpi=config.DPI,
                    bbox_inches="tight")
    plt.close(fig)


def figure2_roc_pr(report_oof):
    ds = next(iter(report_oof))
    y = report_oof[ds]["y"]
    prev = report_oof[ds]["prev"]
    fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
    ax_roc, ax_pr = axes
    for model, proba in report_oof[ds]["models"].items():
        if len(np.unique(y)) >= 2:
            fpr, tpr, _ = roc_curve(y, proba)
            ax_roc.plot(fpr, tpr, lw=1.4, label=config.MODEL_LABELS.get(model, model))
        prec, rec, _ = precision_recall_curve(y, proba)
        ax_pr.plot(rec, prec, lw=1.4, label=config.MODEL_LABELS.get(model, model))
    ax_roc.plot([0, 1], [0, 1], "k--", lw=0.8)
    ax_roc.set_title(f"ROC, {ds}")
    ax_roc.set_xlabel("False positive rate")
    ax_roc.set_ylabel("True positive rate")
    ax_roc.legend(fontsize=7, loc="lower right")
    ax_pr.axhline(prev, ls="--", color="k", lw=0.8, label=f"prevalence={prev:.2f}")
    ax_pr.set_title(f"Precision recall, {ds}")
    ax_pr.set_xlabel("Recall")
    ax_pr.set_ylabel("Precision")
    ax_pr.legend(fontsize=7, loc="upper right")
    fig.tight_layout()
    for ext in ("png", "pdf"):
        fig.savefig(config.FIGDIR / f"figure2_roc_pr.{ext}", dpi=config.DPI,
                    bbox_inches="tight")
    plt.close(fig)


def figure3_confusion_calibration(selected_oof):
    ds, d = next(iter(selected_oof.items()))
    y, pred, proba = d["y"], d["pred"], d["proba"]
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
    cm = confusion_matrix(y, pred, labels=[0, 1])
    ax = axes[0]
    ax.imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=14)
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["non CKD", "CKD"])
    ax.set_yticklabels(["non CKD", "CKD"])
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(f"Confusion, {ds}\n({config.MODEL_LABELS.get(d['model'], d['model'])})")

    ax2 = axes[1]
    frac_pos, mean_pred = calibration_curve(y, proba, n_bins=10, strategy="quantile")
    ax2.plot(mean_pred, frac_pos, "o-", label="model")
    ax2.plot([0, 1], [0, 1], "k--", lw=0.8, label="perfect")
    ax2.set_title(f"Calibration, {ds}")
    ax2.set_xlabel("Mean predicted probability")
    ax2.set_ylabel("Observed frequency")
    ax2.legend(fontsize=8)
    fig.tight_layout()
    for ext in ("png", "pdf"):
        fig.savefig(config.FIGDIR / f"figure3_confusion_calibration.{ext}",
                    dpi=config.DPI, bbox_inches="tight")
    plt.close(fig)


def montage(image_paths, titles, out_name, ncols=1, figsize=(10, 7)):
    pairs = [(p, t) for p, t in zip(image_paths, titles) if p]
    if not pairs:
        return
    paths, titles = zip(*pairs)
    ncols = min(ncols, len(paths))
    nrows = int(np.ceil(len(paths) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize, squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for ax, p, t in zip(axes.ravel(), paths, titles):
        ax.imshow(mpimg.imread(p))
        ax.set_title(t, fontsize=10)
        ax.axis("off")
    fig.tight_layout()
    for ext in ("png", "pdf"):
        fig.savefig(config.FIGDIR / f"{out_name}.{ext}", dpi=config.DPI,
                    bbox_inches="tight")
    plt.close(fig)


def learning_curves(meta, final_model, fname):
    if meta.grouped:
        cv = StratifiedGroupKFold(n_splits=config.OUTER_SPLITS, shuffle=True,
                                  random_state=config.REPORT_SEED)
        splits = list(cv.split(meta.X, meta.y, meta.groups))
    else:
        cv = StratifiedKFold(n_splits=config.OUTER_SPLITS, shuffle=True,
                             random_state=config.REPORT_SEED)
        splits = cv
    scoring = "roc_auc" if meta.primary == "roc_auc" else "average_precision"
    sizes, train_sc, val_sc = learning_curve(
        final_model, meta.X, meta.y, cv=splits, scoring=scoring,
        train_sizes=np.linspace(0.2, 1.0, 5), n_jobs=config.N_JOBS,
        groups=meta.groups if meta.grouped else None)
    fig, ax = plt.subplots(figsize=(6, 4.5))
    ax.plot(sizes, train_sc.mean(1), "o-", label="train")
    ax.plot(sizes, val_sc.mean(1), "s-", label="validation")
    ax.fill_between(sizes, val_sc.mean(1) - val_sc.std(1),
                    val_sc.mean(1) + val_sc.std(1), alpha=0.15)
    ax.set_xlabel("Training examples")
    ax.set_ylabel(scoring)
    ax.set_title(f"Learning curve, {meta.name}")
    ax.legend()
    fig.tight_layout()
    fig.savefig(config.FIGDIR / f"{fname}.png", dpi=config.DPI, bbox_inches="tight")
    plt.close(fig)


Writing report.py


## 5. Run Full Pipeline

In [8]:
"""End to end runner for urinalysis CKD analysis only."""

import json
import time
import pickle
import warnings
import sys
import platform
import hashlib
import importlib.metadata as importlib_metadata
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

import config
import data as datamod
import modeling as M
import explain as X
import report as R

warnings.filterwarnings("ignore")
t0 = time.time()


def log(msg):
    print(f"[{time.time()-t0:7.1f}s] {msg}", flush=True)


def primary_metric_name(meta):
    return "PR_AUC" if meta.primary == "average_precision" else "ROC_AUC"


def json_default(obj):
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        value = float(obj)
        return None if not np.isfinite(value) else value
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (pd.Series, pd.Index)):
        return obj.tolist()
    if isinstance(obj, pd.DataFrame):
        return obj.to_dict(orient="records")
    if isinstance(obj, Path):
        return str(obj)
    return repr(obj)


def write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, default=json_default), encoding="utf-8")


def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def installed_version(package_name):
    try:
        return importlib_metadata.version(package_name)
    except importlib_metadata.PackageNotFoundError:
        return None


def save_requirements_and_environment():
    req_text = config.latest_requirements_text()
    root_req = config.ROOT / "requirements_latest_python312.txt"
    art_req = config.ARTDIR / "requirements_latest_python312.txt"
    root_req.write_text(req_text, encoding="utf-8")
    art_req.write_text(req_text, encoding="utf-8")
    env = {
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "recommended_python": config.PYTHON_RECOMMENDED,
        "package_pin_date": config.PACKAGE_PIN_DATE,
        "python_runtime": sys.version,
        "platform": platform.platform(),
        "package_pins": config.LATEST_PACKAGE_PINS,
        "installed_versions": {pkg: installed_version(pkg) for pkg in config.LATEST_PACKAGE_PINS},
        "requirements_file": str(art_req),
    }
    write_json(config.ARTDIR / "environment_report.json", env)
    pd.DataFrame([
        {"package": pkg, "pinned_version": pin, "installed_version": env["installed_versions"].get(pkg)}
        for pkg, pin in config.LATEST_PACKAGE_PINS.items()
    ]).to_csv(config.TABDIR / "journal_environment_versions.csv", index=False)
    return env


def dataset_fingerprint(meta):
    raw = meta.raw
    target_counts = {}
    if 'd_status' in raw.columns:
        target_counts = {str(k): int(v) for k, v in raw['d_status'].value_counts(dropna=False).to_dict().items()}
    return {
        "dataset": meta.name, "file_path": str(Path(config.DATA).resolve()),
        "file_sha256": sha256_file(config.DATA), "raw_rows": int(raw.shape[0]),
        "raw_columns": int(raw.shape[1]), "processed_rows": int(meta.X.shape[0]),
        "processed_features": int(meta.X.shape[1]), "positive_count": int(meta.y.sum()),
        "negative_count": int(len(meta.y) - meta.y.sum()), "positive_rate": float(np.mean(meta.y)),
        "missing_raw": int(raw.isna().sum().sum()),
        "missing_processed": int(meta.X.isna().sum().sum()), "grouped": bool(meta.grouped),
        "unique_groups": int(len(np.unique(meta.groups))) if meta.groups is not None else None,
        "primary_metric": meta.primary, "raw_target_counts": target_counts,
        "raw_columns_list": list(raw.columns), "processed_feature_order": list(meta.X.columns),
        "continuous_features": list(meta.continuous), "categorical_features": list(meta.categorical),
    }


def save_dataset_and_preprocessing_audit(meta):
    fingerprint = dataset_fingerprint(meta)
    write_json(config.ARTDIR / "dataset_fingerprints.json", [fingerprint])
    pd.DataFrame([{k: v for k, v in fingerprint.items() if not isinstance(v, (list, dict))}]).to_csv(
        config.TABDIR / "journal_dataset_fingerprints.csv", index=False)
    details = {meta.name: {
        "target": config.TARGET_DESCRIPTION,
        "group_column": config.GROUP_COLUMN_DESCRIPTION,
        "removed_predictors": config.REMOVED_PREDICTORS,
        "derived_predictors": config.DERIVED_PREDICTORS,
        "continuous": list(meta.continuous),
        "categorical": list(meta.categorical),
        "imputation": {"continuous": "median inside CV fold", "categorical": "most frequent inside CV fold"},
        "scaling": "StandardScaler only for logistic regression, SVM, and MLP inside CV fold",
        "imbalance": "class weighting for most models and random oversampling for MLP",
    }}
    write_json(config.ARTDIR / "preprocessing_details.json", details)
    feature_rows = []
    for rank, feat in enumerate(meta.X.columns, start=1):
        feature_rows.append({
            "dataset": meta.name, "feature_order": rank, "feature": feat,
            "feature_type": "continuous" if feat in meta.continuous else "categorical",
            "feature_category": meta.feature_categories.get(feat),
            "imputation": "median" if feat in meta.continuous else "most_frequent",
            "scaled_for_models": ", ".join(sorted(config.SCALED_MODELS)) if feat in meta.continuous else "not scaled",
        })
    pd.DataFrame(feature_rows).to_csv(config.TABDIR / "journal_preprocessing_features.csv", index=False)
    return [fingerprint], details


def save_search_space_audit():
    rows = []
    search_space = {}
    for model_name in config.MODELS:
        space = M.param_space(model_name)
        search_space[model_name] = {param: repr(values) for param, values in space.items()}
        for param, values in search_space[model_name].items():
            rows.append({"model": model_name, "parameter": param, "search_values_or_distribution": values})
    write_json(config.ARTDIR / "hyperparameter_search_space.json", search_space)
    pd.DataFrame(rows).to_csv(config.TABDIR / "journal_hyperparameter_search_space.csv", index=False)
    return search_space


def record_fold_audit(rows, split_rows, analysis, meta, model_name, seed, res, elapsed_seconds):
    for item in res.get("fold_summaries", []):
        rows.append({
            "analysis": analysis, "dataset": meta.name, "model": model_name,
            "seed": int(seed), "fold": int(item.get("fold")),
            "primary_metric": primary_metric_name(meta),
            "best_params_json": json.dumps(item.get("best_params", {}), default=json_default),
            "threshold": item.get("threshold"), "train_n": item.get("train_n"),
            "test_n": item.get("test_n"), "train_positive": item.get("train_positive"),
            "train_negative": item.get("train_negative"), "test_positive": item.get("test_positive"),
            "test_negative": item.get("test_negative"), "train_groups": item.get("train_groups"),
            "test_groups": item.get("test_groups"), "tn": item.get("tn"), "fp": item.get("fp"),
            "fn": item.get("fn"), "tp": item.get("tp"),
            "elapsed_seconds_model_seed": elapsed_seconds,
            **{metric: item.get(metric) for metric in R.METRIC_ORDER},
        })
        split_rows.append({
            "analysis": analysis, "dataset": meta.name, "model": model_name,
            "seed": int(seed), "fold": int(item.get("fold")),
            "train_indices": item.get("train_indices", []),
            "test_indices": item.get("test_indices", []),
        })


def save_fold_audit(rows, split_rows):
    df = pd.DataFrame(rows)
    df.to_csv(config.TABDIR / "journal_hyperparameters_per_fold.csv", index=False)
    try:
        df.drop(columns=["best_params_json"], errors="ignore").to_latex(
            config.TABDIR / "journal_hyperparameters_per_fold.tex", index=False, escape=False)
    except Exception:
        pass
    write_json(config.ARTDIR / "nested_cv_split_indices.json", split_rows)
    write_json(config.ARTDIR / "journal_hyperparameters_per_fold.json", rows)
    return df


def save_statistical_tests(meta, store):
    rows = []
    sel = store["selected"]
    pm = primary_metric_name(meta)
    selected_values = pd.DataFrame(store["model_fold_metrics"][sel])[pm].values
    for comp in config.MODELS:
        if comp == sel or comp not in store["model_fold_metrics"]:
            continue
        comp_values = pd.DataFrame(store["model_fold_metrics"][comp])[pm].values
        n = min(len(selected_values), len(comp_values))
        diff = selected_values[:n] - comp_values[:n]
        try:
            if np.allclose(diff, 0):
                statistic, p_value = np.nan, 1.0
            else:
                stat = wilcoxon(selected_values[:n], comp_values[:n], zero_method="wilcox", alternative="two-sided")
                statistic, p_value = float(stat.statistic), float(stat.pvalue)
        except Exception:
            statistic, p_value = np.nan, np.nan
        rows.append({
            "dataset": meta.name, "primary_metric": pm, "selected_model": sel,
            "comparison_model": comp, "mean_selected": float(np.mean(selected_values[:n])),
            "mean_comparison": float(np.mean(comp_values[:n])),
            "mean_difference": float(np.mean(diff)), "wilcoxon_statistic": statistic,
            "wilcoxon_p_value": p_value, "n_paired_folds": int(n),
        })
    df = pd.DataFrame(rows)
    df.to_csv(config.TABDIR / "supp_statistical_tests.csv", index=False)
    return df


def output_manifest():
    rows = []
    for folder, kind in [(config.TABDIR, "table"), (config.FIGDIR, "figure"), (config.ARTDIR, "artifact")]:
        for path in sorted(folder.glob("*")):
            if path.is_file() and path.name not in {"output_manifest.csv", "output_manifest.json"}:
                rows.append({
                    "kind": kind, "relative_path": str(path.relative_to(config.OUT)),
                    "size_bytes": int(path.stat().st_size), "sha256": sha256_file(path),
                })
    df = pd.DataFrame(rows)
    df.to_csv(config.ARTDIR / "output_manifest.csv", index=False)
    write_json(config.ARTDIR / "output_manifest.json", rows)
    return df


log(f"profile = {config.PROFILE}; models = {config.MODELS}; seeds = {config.SEEDS}")
meta = datamod.load_dataset()
log(f"{meta.name}: X={meta.X.shape}, pos={int(meta.y.sum())}/{len(meta.y)}")
DATASETS = [meta]
environment_report = save_requirements_and_environment()
dataset_fingerprints, preprocessing_details = save_dataset_and_preprocessing_audit(meta)
search_space_audit = save_search_space_audit()
runtime_rows = []
fold_audit_rows = []
split_audit_rows = []
final_fit_rows = []
interaction_status_rows = []

log(f"=== {meta.name}: nested CV ===")
model_fold_metrics = {}
report_oof = {}
for name in config.MODELS:
    all_folds = []
    for seed in config.SEEDS:
        start = time.time()
        res = M.run_nested_cv(meta, name, seed, imbalance="balanced")
        elapsed = time.time() - start
        runtime_rows.append({"stage": "nested_cv", "dataset": meta.name, "model": name,
                             "seed": int(seed), "elapsed_seconds": elapsed})
        record_fold_audit(fold_audit_rows, split_audit_rows, "main_nested_cv", meta, name, seed, res, elapsed)
        all_folds.extend(res["fold_metrics"])
        if seed == config.REPORT_SEED:
            report_oof[name] = {"proba": res["oof_proba"], "pred": res["oof_pred"]}
    model_fold_metrics[name] = all_folds
    pm = primary_metric_name(meta)
    log(f"  {name:14s} {pm}={np.mean([f[pm] for f in all_folds]):.3f}")

store = {"meta": meta, "model_fold_metrics": model_fold_metrics, "report_oof": report_oof}
agg = R.aggregate_metrics(model_fold_metrics)
selected, selected_tree = M.select_model(meta, agg)
store.update({"agg": agg, "selected": selected, "selected_tree": selected_tree})
log(f"selected = {selected}; tree for XAI = {selected_tree}")


agg_display = agg.copy()


performance_table = R.performance_table(agg_display, meta.primary, selected, config.PERFORMANCE_TABLE_NAME)
fold_audit_df = save_fold_audit(fold_audit_rows, split_audit_rows)


def pooled_oof(model):
    r = store["report_oof"][model]
    mask = ~np.isnan(r["proba"])
    groups = meta.groups[mask] if meta.groups is not None else None
    return meta.y[mask], r["proba"][mask], r["pred"][mask], groups


def second_best():
    pm = primary_metric_name(meta)
    order = store["agg"][(pm, "mean")].sort_values(ascending=False).index.tolist()
    return next(m for m in order if m != selected)


robust_rows = []
boot_seed_counter = 0
pm = primary_metric_name(meta)
y, proba, pred, groups = pooled_oof(selected)
lo, hi = M.bootstrap_ci(y, proba, pred, groups, pm, seed=config.RNG + boot_seed_counter)
boot_seed_counter += 1
robust_rows.append({"dataset": meta.name, "analysis": f"Selected ({selected}) {pm}",
                    "value": np.mean([f[pm] for f in model_fold_metrics[selected]]),
                    "ci_low": lo, "ci_high": hi})
comparisons = {"selected_vs_logreg": "logreg", "selected_vs_second": second_best()}


for tag, comp in comparisons.items():
    if comp == selected:
        continue
    if comp == "time_only":
        rb = store["time_only_report"]
        mask = ~np.isnan(rb["proba"])
        yc, pc, prc = meta.y[mask], rb["proba"][mask], rb["pred"][mask]
    else:
        yc, pc, prc, _ = pooled_oof(comp)
    diff = M.METRIC_FUNCS[pm](y, proba, pred) - M.METRIC_FUNCS[pm](yc, pc, prc)
    rs = np.random.RandomState(config.RNG + boot_seed_counter)
    boot_seed_counter += 1
    n = min(len(y), len(yc))
    diffs = []
    if groups is not None:
        uniq = np.unique(groups)
        idx_by = {g: np.where(groups == g)[0] for g in uniq}
        for _ in range(config.N_BOOTSTRAP):
            chosen = rs.choice(uniq, size=len(uniq), replace=True)
            idx = np.concatenate([idx_by[g] for g in chosen])
            idx = idx[idx < n]
            if len(np.unique(y[idx])) < 2:
                continue
            diffs.append(M.METRIC_FUNCS[pm](y[idx], proba[idx], pred[idx]) -
                         M.METRIC_FUNCS[pm](yc[idx], pc[idx], prc[idx]))
    else:
        for _ in range(config.N_BOOTSTRAP):
            idx = rs.randint(0, n, n)
            if len(np.unique(y[idx])) < 2:
                continue
            diffs.append(M.METRIC_FUNCS[pm](y[idx], proba[idx], pred[idx]) -
                         M.METRIC_FUNCS[pm](yc[idx], pc[idx], prc[idx]))
    diffs = np.asarray(diffs)
    robust_rows.append({"dataset": meta.name, "analysis": f"{tag} ({pm} diff)",
                        "value": float(diff),
                        "ci_low": float(np.percentile(diffs, 2.5)) if diffs.size else np.nan,
                        "ci_high": float(np.percentile(diffs, 97.5)) if diffs.size else np.nan})
stat_tests_df = save_statistical_tests(meta, store)



calibration = M.calibration_metrics(y, proba)
calibration.update({"dataset": meta.name, "model": config.MODEL_LABELS.get(selected, selected)})
calib_table = pd.DataFrame([calibration]).set_index("dataset")
R._save_table(calib_table, "supp_calibration", float_fmt="%.4f")
selected_oof = {meta.name: {"y": y, "proba": proba, "pred": pred, "model": selected}}

log("=== explainability ===")
res = M.run_nested_cv(meta, selected_tree, config.REPORT_SEED,
                      imbalance="balanced", keep_estimators=True)
shap_summary, beeswarm_path = X.shap_stability(meta, selected_tree,
                                                res["fold_estimators"], res["fold_test_idx"])
perm_summary = X.permutation_summary(meta, res["fold_estimators"], res["fold_test_idx"])
agreement = X.shap_perm_agreement(shap_summary, perm_summary)
final_model, final_metadata = X.fit_final(meta, selected_tree, return_metadata=True)
final_fit_rows.append({"dataset": meta.name, "model": selected_tree, **final_metadata})
pdp_path, pdp_feats = X.pdp_ice(meta, final_model, shap_summary)
rep = report_oof[selected_tree]
local_df, local_paths = X.local_explanations(meta, final_model, rep["proba"], rep["pred"])
interactions, interaction_path, interaction_error = pd.DataFrame(), None, None
interaction_status_rows.append({
    "dataset": meta.name, "status": "not applicable", "path": None,
    "reason": "SHAP interaction analysis was not part of the urinalysis dataset workflow",
})
R._save_table(shap_summary.set_index("feature"), f"supp_shap_summary_{meta.name}", float_fmt="%.4f")
R._save_table(perm_summary.set_index("feature"), f"supp_permutation_{meta.name}", float_fmt="%.4f")

final_fit_df = pd.DataFrame(final_fit_rows)
final_fit_df["best_params_json"] = final_fit_df["best_params"].apply(lambda x: json.dumps(x, default=json_default))
final_fit_df["search_space_json"] = final_fit_df["search_space"].apply(lambda x: json.dumps(x, default=json_default))
final_fit_df.drop(columns=["best_params", "search_space"], errors="ignore").to_csv(
    config.TABDIR / "journal_final_model_parameters.csv", index=False)
write_json(config.ARTDIR / "final_model_parameters.json", final_fit_rows)
interaction_status_df = pd.DataFrame(interaction_status_rows)
interaction_status_df.to_csv(config.TABDIR / "journal_shap_interaction_status.csv", index=False)

R.table1_characteristics(meta)
R.descriptive_stats(meta, config.DESCRIPTIVE_TABLE_NAME)
robust_df = pd.DataFrame(robust_rows)
R._save_table(robust_df.set_index(["dataset", "analysis"]), "table4_robustness", float_fmt="%.4f")
explanation_table = R.table5_explanations(
    shap_summary, perm_summary, agreement, f"table5_explanations_{meta.name}")
if not local_df.empty:
    local_df.to_csv(config.TABDIR / "supp_local_cases.csv", index=False)
else:
    pd.DataFrame(columns=[
        "dataset", "case", "row_index", "true_class", "pred_class", "pred_proba",
        "top_shap_support", "top_shap_oppose", "shap_lime_sign_agreement",
    ]).to_csv(config.TABDIR / "supp_local_cases.csv", index=False)

log("=== figures ===")
R.figure1_workflow(meta)
all_masks = np.stack([~np.isnan(report_oof[name]["proba"]) for name in config.MODELS], axis=0).all(axis=0)
report_oof_fig = {meta.name: {
    "y": meta.y[all_masks],
    "models": {name: report_oof[name]["proba"][all_masks] for name in config.MODELS},
    "prev": float(meta.y.mean()),
}}
R.figure2_roc_pr(report_oof_fig)
R.figure3_confusion_calibration(selected_oof)
R.montage([beeswarm_path], [f"SHAP, {meta.name}"], "figure4_shap_beeswarm", ncols=1, figsize=(10, 7))
R.montage([pdp_path], [f"PDP and ICE, {meta.name}"], "figure5_pdp_ice", ncols=1, figsize=(12, 6))
local_images, local_titles = [], []
for case in ["TP", "FN"]:
    key = f"shap_{case}"
    if key in local_paths:
        local_images.append(local_paths[key])
        local_titles.append(f"{meta.name} {case}")
R.montage(local_images, local_titles, "figure6_local_explanations", ncols=2, figsize=(14, 6))
R.learning_curves(meta, final_model, f"supp_learning_curve_{meta.name}")

params_dump = {meta.name: {"selected": selected, "selected_tree": selected_tree, "primary": meta.primary}}
write_json(config.ARTDIR / "selection.json", params_dump)
slim = {
    meta.name: {
        "agg": agg, "selected": selected, "selected_tree": selected_tree,
        "model_fold_metrics": model_fold_metrics, "shap_summary": shap_summary,
        "perm_summary": perm_summary, "agreement": agreement, "local_df": local_df,
        "interactions": interactions, "final_fit_metadata": final_metadata,
        "interaction_error": interaction_error,
    },
    "robustness": robust_df, "calibration": calib_table,
}
with open(config.ARTDIR / "results.pkl", "wb") as f:
    pickle.dump(slim, f)
pd.DataFrame(runtime_rows).to_csv(config.TABDIR / "supp_runtime.csv", index=False)
reproducibility_report = {
    "created_utc": datetime.now(timezone.utc).isoformat(), "profile": config.PROFILE,
    "seeds": config.SEEDS, "rng": config.RNG, "n_jobs": config.N_JOBS,
    "outer_splits": config.OUTER_SPLITS, "inner_splits": config.INNER_SPLITS,
    "environment": environment_report, "dataset_fingerprints": dataset_fingerprints,
    "preprocessing_details": preprocessing_details, "search_space": search_space_audit,
    "selection": params_dump, "final_model_parameters": final_fit_rows,
    "shap_interaction_status": interaction_status_rows,
    "statistical_tests_file": str(config.TABDIR / "supp_statistical_tests.csv"),
    "fold_hyperparameter_file": str(config.TABDIR / "journal_hyperparameters_per_fold.csv"),
    "split_indices_file": str(config.ARTDIR / "nested_cv_split_indices.json"),
}
write_json(config.ARTDIR / "reproducibility_report.json", reproducibility_report)
manifest_df = output_manifest()

print("\n" + "=" * 70)
print("SELECTED MODEL")
print("=" * 70)
print(f"{meta.name}: {selected} (XAI tree: {selected_tree})")
print("\nPERFORMANCE TABLE")
print(performance_table.to_string())
print("\nROBUSTNESS TABLE")
print(robust_df.to_string(index=False))
print("\nCALIBRATION")
print(calib_table.to_string())
print("\nTOP 10 SHAP FEATURES")
cols_show = ["mean_abs_shap", "std_abs_shap", "median_rank", "top10_frequency", "category"]
print(shap_summary.head(10).set_index("feature")[cols_show].to_string())
print("\nSHAP AND PERMUTATION AGREEMENT")
print(agreement)
print("\nSAVED TABLES")
for p in sorted(config.TABDIR.glob("*")):
    print(f"  {p.name}")
print("\nSAVED FIGURES")
for p in sorted(config.FIGDIR.glob("*")):
    print(f"  {p.name}")
print("\nSAVED ARTIFACTS")
for p in sorted(config.ARTDIR.glob("*")):
    print(f"  {p.name}")
print(f"\nRun complete in {time.time()-t0:.0f}s")

CKD profile: paper
Dataset path: /kaggle/input/datasets/miftahuladib/datasets-ckd/CKD_dataset.xlsx
Output path: /kaggle/working/outputs_urinalysis
N_JOBS: 1
[    0.0s] profile = paper; models = ['logreg', 'decision_tree', 'random_forest', 'xgboost', 'lightgbm', 'catboost', 'svm', 'mlp']; seeds = [0, 1, 2]
[    1.0s] Dataset1_Urinalysis: X=(380, 20), pos=239/380
[    1.0s] === Dataset1_Urinalysis: nested CV ===
[   30.3s]   logreg         ROC_AUC=0.973
[   48.8s]   decision_tree  ROC_AUC=0.924
[  450.8s]   random_forest  ROC_AUC=0.973
[  602.9s]   xgboost        ROC_AUC=0.970
[  701.9s]   lightgbm       ROC_AUC=0.966
[ 2060.9s]   catboost       ROC_AUC=0.969
[ 2085.5s]   svm            ROC_AUC=0.967
[ 2171.1s]   mlp            ROC_AUC=0.959
[ 2171.1s] selected = random_forest; tree for XAI = random_forest
[ 2177.2s] === explainability ===
[ 2385.1s] === figures ===

SELECTED MODEL
Dataset1_Urinalysis: random_forest (XAI tree: random_forest)

PERFORMANCE TABLE
                           

## 6. Save Raw and Preprocessed Dataset

In [9]:
from pathlib import Path
import config

raw_dir = config.OUT / "raw_data"
prep_dir = config.OUT / "preprocessed_data"
raw_dir.mkdir(parents=True, exist_ok=True)
prep_dir.mkdir(parents=True, exist_ok=True)

meta.raw.to_excel(raw_dir / Path(config.DATA).name, index=False)
prep = meta.X.copy()
prep["target"] = meta.y
prep.to_csv(prep_dir / f"{meta.name}_preprocessed.csv", index=False)

print(f"Raw dataset saved to: {raw_dir}")
print(f"Preprocessed dataset saved to: {prep_dir}")
for p in sorted(raw_dir.glob("*")):
    print(f"  raw_data/{p.name}")
for p in sorted(prep_dir.glob("*")):
    print(f"  preprocessed_data/{p.name}")


Raw dataset saved to: /kaggle/working/outputs_urinalysis/raw_data
Preprocessed dataset saved to: /kaggle/working/outputs_urinalysis/preprocessed_data
  raw_data/CKD_dataset.xlsx
  preprocessed_data/Dataset1_Urinalysis_preprocessed.csv
